# XGBoost：Train-fitted One-Hot Boosting 挑战者

本 Notebook 使用与共同输入 Logistic、随机森林、CatBoost 和 LightGBM 完全相同的 126 个公共特征和固定 Train/Selection 切分。数值缺失保留为 `NaN`，使 XGBoost 区分真实 0 与缺失；15 个物理类别变量只在 Train 上拟合密集 One-Hot 编码，Selection 未见类别在对应 dummy 组全为 0。模型使用 `hist + depthwise`，候选方案覆盖深度、最小 Hessian 叶约束、学习率、L1/L2、行列抽样和最小分裂收益。Selection 仅用于 early stopping、锁模和与共同输入 Logistic 的配对比较，Test 保持封存。Tree SHAP 会先合并同一原始类别变量的 dummy 贡献。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'configs' / 'experiment.yaml').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.comparison import load_locked_selection_prediction
from credit_risk.development import load_development_data
from credit_risk.models.xgboost import run_xgboost
print('Project root:', PROJECT_ROOT)

Project root: /Users/ganlu/Desktop/home-credit-risk-english


In [2]:
development = load_development_data(PROJECT_ROOT)
assert not hasattr(development, 'X_test')
assert not hasattr(development, 'y_test')
reference_selection_prediction = load_locked_selection_prediction(
    development, 'common_logistic'
)
display(development.split_overview)
print('公共特征数:', len(development.feature_columns))
print('已对齐的共同输入 Logistic Selection 预测数:', len(reference_selection_prediction))

,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test (sealed),46127,NaN


公共特征数: 126
已对齐的共同输入 Logistic Selection 预测数: 46127


In [3]:
search_settings = development.configs['model_search_spaces']['xgboost']
display(pd.DataFrame(search_settings['candidate_profiles']))
print('基础迭代上限:', search_settings['max_iterations'])
print('边界延长上限:', search_settings['extension_max_iterations'])
print('Early stopping patience:', search_settings['early_stopping_rounds'])

,name,max_depth,min_child_weight,learning_rate,reg_lambda,reg_alpha,subsample,colsample_bytree,gamma
0,conservative_depth3,3,20.0,0.02,10.0,0.0,0.7,0.8,0.00
1,shallow_stable,3,10.0,0.03,5.0,0.0,0.8,0.9,0.00
2,legacy_balanced,4,10.0,0.03,5.0,0.0,0.8,0.9,0.00
3,regularized_depth4,4,20.0,0.03,10.0,1.0,0.8,0.9,0.05
4,depth5_regularized,5,20.0,0.03,20.0,1.0,0.8,0.8,0.05
5,faster_strong_l2,4,20.0,0.05,10.0,1.0,0.8,0.9,0.05
6,deep_conservative,5,40.0,0.02,20.0,1.0,0.7,0.8,0.10


基础迭代上限: 3000
边界延长上限: 5000
Early stopping patience: 150


In [4]:
result = run_xgboost(
    development,
    reference_selection_prediction=reference_selection_prediction,
    register_lock=True,
)
print('XGBoost 搜索、固定迭代重训与锁模完成。')

XGBoost candidate 1 [base_search/conservative_depth3] Selection AR=0.532956, gap=0.060786, trees=2796/2946, fit=74.9s


XGBoost candidate 2 [base_search/shallow_stable] Selection AR=0.534463, gap=0.069835, trees=2095/2245, fit=57.5s


XGBoost candidate 3 [base_search/legacy_balanced] Selection AR=0.533791, gap=0.098836, trees=1417/1567, fit=43.1s


XGBoost candidate 4 [base_search/regularized_depth4] Selection AR=0.534267, gap=0.091486, trees=1437/1587, fit=49.6s


XGBoost candidate 5 [base_search/depth5_regularized] Selection AR=0.533080, gap=0.109297, trees=1051/1201, fit=42.5s


XGBoost candidate 6 [base_search/faster_strong_l2] Selection AR=0.532207, gap=0.094450, trees=870/1020, fit=33.5s


XGBoost candidate 7 [base_search/deep_conservative] Selection AR=0.533309, gap=0.096128, trees=1540/1690, fit=63.1s


XGBoost 搜索、固定迭代重训与锁模完成。


In [5]:
display(result.search_results)
display(result.metrics)
display(result.uplift_vs_logistic)
display(result.raw_feature_importance.head(30))
display(result.shap_importance.head(30))
display(result.selection_deciles)
display(result.locked_config)
print('锁模产物:', result.run_directory)

,candidate,stage,profile,max_iterations,attempted_iterations,best_iteration,reached_iteration_boundary,max_depth,min_child_weight,learning_rate,...,selection_ar,ar_gap,train_ks,selection_ks,selection_ap,selection_logloss,capacity,fit_seconds,parameters_json,eligible_for_lock
0,2,base_search,shallow_stable,3000,2245,2095,False,3,10.0,0.03,...,0.534463,0.069835,0.456573,0.395173,0.257843,0.243287,1206.720,57.535768,"{""colsample_bytree"": 0.9, ""device"": ""cpu"", ""ea...",True
1,4,base_search,regularized_depth4,3000,1587,1437,False,4,20.0,0.03,...,0.534267,0.091486,0.472882,0.395154,0.257389,0.243298,827.712,49.605712,"{""colsample_bytree"": 0.9, ""device"": ""cpu"", ""ea...",True
2,3,base_search,legacy_balanced,3000,1567,1417,False,4,10.0,0.03,...,0.533791,0.098836,0.479956,0.394407,0.257132,0.243385,1632.384,43.094183,"{""colsample_bytree"": 0.9, ""device"": ""cpu"", ""ea...",True
3,7,base_search,deep_conservative,3000,1690,1540,False,5,40.0,0.02,...,0.533309,0.096128,0.475227,0.393740,0.255949,0.243481,689.920,63.129405,"{""colsample_bytree"": 0.8, ""device"": ""cpu"", ""ea...",True
4,5,base_search,depth5_regularized,3000,1201,1051,False,5,20.0,0.03,...,0.533080,0.109297,0.486959,0.392876,0.257595,0.243401,1076.224,42.504323,"{""colsample_bytree"": 0.8, ""device"": ""cpu"", ""ea...",True
5,1,base_search,conservative_depth3,3000,2946,2796,False,3,20.0,0.02,...,0.532956,0.060786,0.447781,0.394147,0.257549,0.243397,626.304,74.880932,"{""colsample_bytree"": 0.8, ""device"": ""cpu"", ""ea...",True
6,6,base_search,faster_strong_l2,3000,1020,870,False,4,20.0,0.05,...,0.532207,0.094450,0.473421,0.391526,0.255044,0.243641,501.120,33.504405,"{""colsample_bytree"": 0.9, ""device"": ""cpu"", ""ea...",True


,sample,n,bad_rate,AUC,AR,KS,AP,Logloss
0,Train,215257,0.080727,0.796871,0.593742,0.447781,0.295337,0.234371
1,Selection,46127,0.080734,0.766478,0.532956,0.394147,0.257549,0.243397


,model,reference_model,metric,difference,lower,upper,bootstrap_se,confidence_level,bootstrap_replicates,higher_is_better
0,xgboost,common_logistic,AUC,0.018335,0.015063,0.021766,0.001789,0.95,500,True
1,xgboost,common_logistic,AR,0.036670,0.030126,0.043533,0.003578,0.95,500,True
2,xgboost,common_logistic,KS,0.027669,0.017715,0.037577,0.005235,0.95,500,True
3,xgboost,common_logistic,AP,0.022120,0.016257,0.028703,0.003143,0.95,500,True
4,xgboost,common_logistic,Logloss,-0.005279,-0.006226,-0.004368,0.000482,0.95,500,False


,raw_feature,importance
0,ORGANIZATION_TYPE,0.116325
1,OCCUPATION_TYPE,0.074559
2,NAME_EDUCATION_TYPE,0.059018
3,EXT_SOURCE_2,0.050045
4,EXT_SOURCE_3,0.039732
5,NAME_INCOME_TYPE,0.030168
6,NAME_HOUSING_TYPE,0.022595
7,NAME_FAMILY_STATUS,0.022409
8,WEEKDAY_APPR_PROCESS_START,0.021483
9,WALLSMATERIAL_MODE,0.021181


,feature,mean_abs_shap
0,EXT_SOURCE_3,0.354399
1,EXT_SOURCE_2,0.311550
2,EXT_SOURCE_1,0.176703
3,CREDIT_ANNUITY_RATIO,0.165463
4,AMT_ANNUITY,0.111187
5,GOODS_CREDIT_RATIO,0.109717
6,AMT_GOODS_PRICE,0.103184
7,AGE_YEARS,0.085476
8,OWN_CAR_AGE,0.084957
9,EMPLOYED_YEARS,0.079320


,risk_group,rank_min,rank_max,n,bad_n,bad_rate,probability_min,probability_max,lift,cumulative_bad_capture
0,1,1,4613,4613,1295,0.280728,0.181818,0.835700,3.477217,0.347744
1,2,4614,9226,4613,662,0.143507,0.119346,0.181771,1.777543,0.525510
2,3,9227,13839,4613,488,0.105788,0.087064,0.119344,1.310334,0.656552
3,4,13840,18451,4612,378,0.081960,0.066667,0.087062,1.015192,0.758056
4,5,18452,23064,4613,291,0.063083,0.052160,0.066662,0.781367,0.836198
5,6,23065,27677,4613,194,0.042055,0.041089,0.052155,0.520911,0.888292
6,7,27678,32289,4612,169,0.036644,0.032228,0.041084,0.453882,0.933673
7,8,32290,36902,4613,122,0.026447,0.024299,0.032228,0.327583,0.966434
8,9,36903,41515,4613,74,0.016042,0.016670,0.024298,0.198698,0.986305
9,10,41516,46127,4612,51,0.011058,0.002336,0.016669,0.136970,1.000000


{'model_name': 'xgboost',
 'experiment_track': 'controlled',
 'data_sha256': '52e96b895b1112e1c853f670e58372719c8441c5ed1c57ac2f7fad559d784f5f',
 'split_sha256': '4ae4ceee661ba8c210fa0a1cf950448799a0aef468659e6d2590c1bb4e8a3835',
 'feature_policy_version': 1,
 'training_sample': 'Train only',
 'selection_role': 'early stopping, candidate selection, and paired comparison only',
 'test_accessed': False,
 'parameters': {'objective': 'binary:logistic',
  'eval_metric': 'auc',
  'n_estimators': 2796,
  'max_depth': 3,
  'min_child_weight': 20.0,
  'learning_rate': 0.02,
  'reg_lambda': 10.0,
  'reg_alpha': 0.0,
  'subsample': 0.7,
  'colsample_bytree': 0.8,
  'gamma': 0.0,
  'tree_method': 'hist',
  'device': 'cpu',
  'grow_policy': 'depthwise',
  'max_bin': 256,
  'missing': nan,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': 0,
  'early_stopping_rounds': None},
 'selected_profile': 'conservative_depth3',
 'candidate_count': 7,
 'base_candidate_count': 7,
 'boundary_extensions': 0,
 

锁模产物: /Users/ganlu/Desktop/home-credit-risk-english/artifacts/xgboost/20260831T070615_394827Z


In [6]:
registry_path = PROJECT_ROOT / development.configs['experiment']['test_policy']['registry_path']
registry = json.loads(registry_path.read_text(encoding='utf-8'))
assert registry['models']['xgboost']['locked'] is True
assert registry['models']['xgboost']['config']['test_accessed'] is False
assert registry['suite_locked'] is False
print('XGBoost 已锁定；MLP 尚未登记，Test 继续封存。')

XGBoost 已锁定；MLP 尚未登记，Test 继续封存。
